# 36쪽 운영 편람 · 검색 결과에서 근거를 고르는 인수 시험

24장에서 만든 문서 챗봇이 새로운 PDF도 읽는지 검사합니다. 먼저 모델 없이 PDF의 실제 문단을 찾고 문의자에게 적용할 근거를 판정합니다. 이어 같은 질문으로 챗봇을 실행해 검색에 실패했는지 답 작성에 실패했는지 가릅니다. 처음부터 끝까지 실행하는 기본 실습은 인터넷과 API 키가 필요하지 않습니다.

[실습 안내](handbook-lab.md) · [원문 PDF](operations.pdf) · [쪽별 정보](pages.csv) · [문제 CSV](questions.csv)

## 준비 · 문의와 쪽 정보를 따로 읽기

실습 묶음을 모두 푼 뒤 이 노트북을 여세요. questions.csv에는 정답이 없습니다. pages.csv는 각 쪽의 문서 종류·지점·대상·유효기간입니다. page-text.json은 제공된 PDF 36쪽에서 가져온 텍스트입니다. text에는 한 쪽 전체, passage에는 각 쪽의 본문이나 접수대장 소개 문단, title에는 쪽 제목이 들어 있습니다. 원문에 표가 있거나 문장이 접혀 있으면 PDF에서 한번 더 확인합니다. 폴더 위치를 찾는 아래 준비 셀은 그대로 실행해도 됩니다.

In [1]:
from pathlib import Path
import csv
import json

# 압축을 푼 자료 폴더와 저장소의 배포 원본 양쪽에서 같은 파일을 찾습니다.
starts = [Path.cwd(), *Path.cwd().parents]
routes = [Path("."), Path("pre-mid/documents/transfer"), Path("site/pre-mid/documents/transfer")]
folder = next((start / route for start in starts for route in routes
               if (start / route / "handbook-evidence.ipynb").is_file()
               and (start / route / "page-text.json").is_file()), None)
if folder is None:
    raise FileNotFoundError("자료 묶음을 모두 풀고 transfer 폴더에서 노트북을 여세요.")

def read_table(name):
    with (folder / name).open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))

inquiries = {row["case_id"]: row for row in read_table("questions.csv")}
catalog = read_table("pages.csv")
content = json.loads((folder / "page-text.json").read_text(encoding="utf-8"))
pages = {row["page"]: row for row in content}
print(len(inquiries), "건의 문의,", len(catalog), "쪽의 메타데이터,", len(pages), "쪽의 본문")

10 건의 문의, 36 쪽의 메타데이터, 36 쪽의 본문


10건과 36쪽·36쪽이 짝을 이룹니다. 페이지 번호는 PDF에서 학생이 누르는 1쪽부터 셉니다. CSV로 읽은 페이지 번호는 문자열이고, JSON 본문에서 읽은 페이지 번호는 정수입니다. 하나의 자료에서 찾은 쪽을 다른 자료에서 열 때는 같은 번호인지 확인하세요.

## 조사 1 · ‘부산’과 ‘90일’이 나오면 근거가 될까요?

먼저 B03 질문을 읽고, ‘부산’과 ‘90일’이 모두 있는 쪽을 찾아봅니다. 이번 검색은 문자 그대로의 일치를 찾기 때문에 몇 쪽이 돌아왔는지와 답할 수 있는지는 별개입니다.

In [2]:
case = inquiries["B03"]
print(case["branch"], case["audience"], case["as_of"], case["question"])
for page in content:
    if "부산" in page["text"] and "90일" in page["text"]:
        print(page["page"], page["title"])

부산 개인 2026-09-01 승인 영수증의 열람 기한은?
4 상담 화면 디자인 회의
8 지역 체험 부스 운영


후보에는 상담 화면 디자인 회의가 포함됩니다. 원문에서는 레이아웃을 검토하면서 부산에도 임시로 90일을 적었고 확정되지 않았다고 설명합니다. 상담사가 화면에서 본 숫자의 출처를 찾았더라도 이용자에게 적용 가능한 정책인지 따로 판정해야 합니다.

In [3]:
print(pages[4]["passage"])  # PDF의 본문을 읽어 시안인지 확인합니다.

연수용 모형 화면에는 서울 90일, 부산 90일을 같은 글꼴로 배치했다. 레이아웃 검토용 예시이며 부산 문안은 확정되지 않았다.


‘확정되지 않았다’는 구절을 표시하세요. B03에서 필요한 것은 승인 영수증을 언제까지 열람할 수 있는지 정한 규정입니다. 같은 지점·대상·적용일의 문서 중 종류가 정책인 페이지로 좁혀 봅니다.

In [4]:
def policies_for(case):
    matches = []
    for entry in catalog:
        same_service = entry["branch"] == case["branch"] and entry["audience"] == case["audience"]
        # YYYY-MM-DD 형식으로 고정된 자료이므로 문자열 순서와 날짜 순서가 같습니다.
        active = entry["valid_from"] <= case["as_of"] <= entry["valid_to"]
        if same_service and active and entry["kind"] == "정책":
            matches.append(int(entry["page"]))
    return matches

for number in policies_for(inquiries["B03"]):
    print(number, pages[number]["passage"])

6 부산 개인 앱의 승인 영수증은 결제일로부터 60일 동안 열람한다. 지점이 다른 문의를 이 표로 처리할 수 없다.


6쪽에서 결제일로부터 60일이라고 확인했습니다. 이 편람에서는 지점·대상이 명시된 정책이 평가 대상입니다. 다른 문서에서 전국 공통 정책이 실제로 존재하면 적용 범위 처리도 달라질 수 있습니다. 6쪽을 PDF로 열고 답에 필요한 기준시점인 ‘결제일로부터’를 밑줄 그으세요.

## 실습 1 · 개정 전날과 개정 당일

B01과 B02는 같은 질문이고 같은 지점입니다. 두 날짜를 먼저 읽고 어떤 쪽을 봐야 할지 손으로 적으세요. policies_for로 찾은 쪽에서 원문 한 줄을 출력한 다음, 날짜가 하루 바뀌어도 유지되는 조건과 바뀌는 답을 구별하세요.

In [5]:
# inquiries["B01"]과 inquiries["B02"]에서 각각 정책 후보를 찾고 본문을 읽으세요.

<details><summary>모범 코드·실행 결과·해설</summary><pre><code>for case_id in [&quot;B01&quot;, &quot;B02&quot;]:
    case = inquiries[case_id]
    for number in policies_for(case):
        print(case_id, case[&quot;as_of&quot;], number, pages[number][&quot;passage&quot;])</code></pre><pre>B01 2026-06-30 1 결제일을 기준으로 개인 앱에서 승인 영수증을 30일 동안 열람한다. 출력한 날에 카운터가 다시 시작되지 않는다.
B02 2026-07-01 7 서울 개인 앱에서는 결제일로부터 90일 동안 승인 영수증을 열람할 수 있다. 개정 기준일은 2026년 7월 1일이다.</pre><p>서울 개인 앱과 기산일인 결제일은 같습니다. 6월 30일은 1쪽의 30일, 7월 1일은 7쪽의 90일입니다. 날짜가 유효기간의 양 끝일 때도 해당 쪽을 포함해야 합니다.</p></details>

## 실습 2 · 같은 날짜에 유효한 두 페이지 중 어느 쪽을 쓸까요?

B08과 B09에 대해 후보 쪽과 본문을 모두 확인하세요. 적합한 페이지가 여러 장일 때 첫 번째 것만 고르는 코드를 쓰면 어느 질문이 틀리는지 먼저 예상합니다. 두 질문이 요구하는 행동을 한 단어씩 붙인 뒤 근거를 선택합니다.

In [6]:
# 하나는 파일이 제공될 시각, 다른 하나는 건수가 맞지 않을 때 처리할 일입니다.

<details><summary>모범 코드·실행 결과·해설</summary><pre><code>for case_id in [&quot;B08&quot;, &quot;B09&quot;]:
    case = inquiries[case_id]
    print(case_id, case[&quot;question&quot;], policies_for(case))
for number in policies_for(inquiries[&quot;B08&quot;]):
    print(number, pages[number][&quot;passage&quot;])</code></pre><pre>B08 파일이 준비되는 시각은? [15, 17]
B09 승인 건수와 파일 행 수가 다르면? [15, 17]
15 승인 건수와 파일 행 수가 다르면 검토 상태로 분리한다. 이 절차는 파일의 준비 시각을 바꾸는 공지가 나와도 계속 적용한다.
17 대전 가맹점 정산 파일의 준비 시각만 다음 영업일 09:30으로 바꾼다. 기존 11:00은 2026년 12월 31일까지 적용한다.</pre><p>두 입력 모두 15쪽과 17쪽의 적용 조건에 맞습니다. 제공 시각은 17쪽의 다음 영업일 09:30, 행 수 불일치는 15쪽의 검토 상태입니다. 15쪽만 선택하면 B08이, 최신 17쪽만 선택하면 B09가 틀립니다. B07까지 함께 시험하면 날짜 경계가 바뀌어 14쪽의 이전 시각이 적용됩니다.</p></details>

## 실습 3 · 돌아온 문서를 근거로 쓰지 않아야 할 때

B10에 대해 두 가지를 확인하세요. 제주라는 글자가 실제 어느 쪽에 있는지, 제주 개인 앱의 적용 가능한 정책이 있는지입니다. 마지막 답으로 확정된 기한을 말할 수 있는지 판단하고 담당자에게 어떤 문서가 필요한지 적으세요.

In [7]:
# 제주가 등장하는 교육 문서와 정책 후보를 각각 조회해 보세요.

<details><summary>모범 코드·실행 결과·해설</summary><pre><code>case = inquiries[&quot;B10&quot;]
for entry in catalog:
    if entry[&quot;branch&quot;] == case[&quot;branch&quot;] and entry[&quot;audience&quot;] == case[&quot;audience&quot;]:
        number = int(entry[&quot;page&quot;])
        print(entry[&quot;kind&quot;], number, pages[number][&quot;passage&quot;])
print(&quot;적용 가능한 정책&quot;, policies_for(case))</code></pre><pre>교육 22 교육 담당자가 서울 문안의 90일을 제주용 화면에 임시로 넣었다. 제주 영수증 열람 기한은 이 문서에서 승인되지 않았다.
적용 가능한 정책 []</pre><p>22쪽에 제주와 90일이 함께 있으나 교육용 초안에 임시로 적은 값입니다. 빈 정책 후보는 이 편람으로 확정할 근거가 없음을 뜻합니다. 제주 개인 앱 서비스 자체가 없다고 판단할 증거도 없습니다.</p></details>

## 인수 시험 · 24장 챗봇이 받은 입력과 답을 보내기

24장의 PDF 업로드에 operations.pdf를 넣습니다. B02·B03·B08·B10 한 건씩 지점·대상·기준일을 포함해 보냅니다. 검색된 페이지, 실제 답변의 핵심값, 인용문과 보류 여부를 복사해 두세요. 예를 들어 B03에서 6쪽을 찾지 못했다면 검색을, 6쪽을 찾고도 90일을 말했다면 모델에 보낸 문단과 지시를 먼저 살핍니다. 출처가 녹음기 설명서라면 새 파일이 실제로 색인에 반영됐는지 확인하세요.

아래 표를 마크다운 셀로 복사해 직접 작성하세요. 값이 서로 다른 이유가 문서 선택인지 단위나 기준 시각을 빠뜨린 것인지 한 문장으로 남깁니다.

| 문의 | 실제 후보 쪽 | 챗봇 답과 실제 인용 | 직접 확인한 문장 | 최초 실패 지점 |
| --- | --- | --- | --- | --- |
| B02 | | | | |
| B03 | | | | |
| B08 | | | | |
| B10 | | | | |

나머지 여섯 문의를 짝이 새 입력으로 맡습니다. 한 번에 한 가지 설정을 고쳐 다시 실행하고, 원래 맞던 질문이 새로 틀렸는지도 비교하세요. 모두 기록한 뒤에만 [기준 답과 반례 해설](handbook-key.md)과 [기준 답 CSV](reference-answers.csv)를 엽니다.